<!-- SPDX-License-Identifier: AGPL-3.0-or-later -->
<!-- Commercial license available -->
<!-- Copyright 1998-2026 Miroslav Sotek. All rights reserved. -->

# SHD Vertex Deployable-Selector Evidence

This notebook summarises downloaded SC-NeuroCore SHD Vertex artifacts for the DCLS-max standard-LIF lane. It exists to keep the collaboration evidence reproducible without turning partial local downloads into final claims.

## Evidence Boundary

This notebook reports only artifact directories present under `results/vertex/director_ai_tim_20260506/` at execution time.

It proves:

- which seed artifacts are locally available;
- deployable-selector accuracy values stored in each `config.json`;
- rounding-drop values stored in each `config.json`;
- aggregate statistics over the available local artifacts.

It does not claim that every intended Vertex seed has been downloaded, that Tim's team has accepted these results, or that physical FPGA measurements have been made.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import statistics
import sys
from pathlib import Path
from types import ModuleType
from typing import Any

REPO = Path.cwd()
if not (REPO / "tools" / "summarise_shd_vertex_runs.py").exists():
    REPO = REPO.parent

RUN_ROOT = REPO / "results/vertex/director_ai_tim_20260506"
TOOL = REPO / "tools/summarise_shd_vertex_runs.py"

def load_summary_tool(path: Path) -> ModuleType:
    spec = importlib.util.spec_from_file_location("summarise_shd_vertex_runs", path)
    assert spec is not None
    assert spec.loader is not None
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module

summary_tool = load_summary_tool(TOOL)
print(f"Run root: {RUN_ROOT}")

## 1. Artifact Inventory

Expected seeds are `0..4` for this local status view. Missing seeds mean missing local artifacts, not failed science.

In [ ]:
expected_seeds = set(range(5))
summary = summary_tool.summarise_runs(RUN_ROOT)
runs: list[dict[str, Any]] = summary["runs"]
observed_seeds = {run["seed"] for run in runs if run["seed"] is not None}
missing_seeds = sorted(expected_seeds - observed_seeds)

assert summary["schema_version"] == summary_tool.SCHEMA_VERSION
assert summary["run_count"] == len(runs)
assert summary["run_count"] > 0

print("available seeds:", sorted(observed_seeds))
print("missing expected seeds:", missing_seeds)
print("available artifact directories:")
for run in runs:
    print("-", run["run"])

## 2. Deployable-Selector Results

`fpga_deployable_test_acc` is the value selected under the deployable rounded-delay condition recorded in each run config.

In [ ]:
deployable_values = [
    float(run["best_fpga_deployable_test_acc"])
    for run in runs
    if run["best_fpga_deployable_test_acc"] is not None
]
rounding_drops = [float(run["rounding_drop"]) for run in runs if run["rounding_drop"] is not None]

assert len(deployable_values) == summary["run_count"]
assert len(rounding_drops) == summary["run_count"]
assert all(0.0 <= value <= 100.0 for value in deployable_values)

sample_std = statistics.stdev(deployable_values) if len(deployable_values) > 1 else 0.0
aggregate = {
    "available_runs": summary["run_count"],
    "available_seeds": sorted(observed_seeds),
    "missing_expected_seeds": missing_seeds,
    "deployable_test_mean": statistics.mean(deployable_values),
    "deployable_test_sample_std": sample_std,
    "deployable_test_min": min(deployable_values),
    "deployable_test_max": max(deployable_values),
    "zero_rounding_drop_runs": sum(1 for value in rounding_drops if value == 0.0),
}

print(json.dumps(aggregate, indent=2))

## 3. Per-Run Table

The table below is generated from local `config.json` and `training_log.csv` files. It should be regenerated after additional Vertex artifacts are downloaded.

In [ ]:
table_rows = []
for run in runs:
    row = {
        "seed": run["seed"],
        "deployable_test": run["best_fpga_deployable_test_acc"],
        "rounding_drop": run["rounding_drop"],
        "best_fpga_epoch": run["best_fpga_deployable_epoch"],
        "last_epoch": run["last_epoch"],
        "last_test_after_round": run["last_test_after_round"],
    }
    table_rows.append(row)

print("| Seed | Deployable test | Rounding drop | Best deployable epoch | Last epoch | Last rounded test |")
print("|---:|---:|---:|---:|---:|---:|")
for row in table_rows:
    print(
        f"| {row['seed']} | {row['deployable_test']:.4f} | {row['rounding_drop']:.4f} | "
        f"{row['best_fpga_epoch']} | {row['last_epoch']} | {row['last_test_after_round']:.4f} |"
    )

## 4. Handoff Manifest

This manifest is the internal handoff object for the collaboration status. It is intentionally tied to local artifacts and should be regenerated after downloads change.

In [ ]:
manifest = {
    "schema_version": "sc-neurocore.shd-vertex-notebook-evidence.v1",
    "source_summary_schema": summary["schema_version"],
    "run_root": str(RUN_ROOT.relative_to(REPO)),
    "aggregate": aggregate,
    "runs": table_rows,
    "claim_boundary": (
        "Downloaded artifact summary only; no claim that all intended seeds are present, "
        "no physical FPGA measurement, and no external acceptance claim."
    ),
}

assert manifest["aggregate"]["available_runs"] == len(manifest["runs"])
assert manifest["claim_boundary"].startswith("Downloaded artifact summary only")

print(json.dumps(manifest, indent=2))